# Topic: Seq2Seq Models & Context Vectors

## Definition (30-second explanation)
- An architecture designed to map an input sequence to an output sequence of a different length.
- **Encoder:** Processes the input sequence and compresses its information into a single, fixed-length representation called the **Context Vector**.
- **Decoder:** Takes this context vector and generates the output sequence token by token.

## Why Interviewers Ask This
- It is the foundational architecture for all modern NLP tasks (translation, summarization).
- Interviewers want to test your understanding of the **Information Bottleneck**, which is the exact problem the Attention mechanism (and later Transformers) was invented to solve.

## Core Concepts
- **Encoder:** Typically an RNN, LSTM, or GRU. It reads the input sequence sequentially.
- **Context Vector:** The final hidden state of the Encoder. It acts as the initial hidden state for the Decoder.
- **Information Bottleneck:** Compressing a long sequence (e.g., a 50-word sentence) into a single fixed-size array causes the model to "forget" earlier tokens, degrading performance on long texts.
- **Teacher Forcing:** A training technique where the Decoder is fed the *actual* target token from the training data as its next input, rather than its own previous prediction, to speed up convergence.

## When to Use
- **Historically:** Machine Translation, Text Summarization, Conversational Agents.
- **Modern Context:** Pure Seq2Seq (without attention) is rarely used in modern production systems; it is primarily a stepping stone to understanding Attention and Transformers.

## Advantages
- Can handle variable-length input sequences and variable-length output sequences (unlike standard feed-forward networks or basic RNNs).
- Requires minimal domain-specific feature engineering (end-to-end learning).

## Limitations
- **Information Bottleneck:** The single context vector struggles to capture the meaning of long sequences.
- **Vanishing Gradients:** Because it relies on recurrent sequential processing, it suffers from standard RNN/LSTM memory issues.
- **Sequential Computation:** Cannot be parallelized during training (unlike Transformers), leading to slow training times.

## Common Comparisons
- **Seq2Seq vs. Standard RNN:** Standard RNNs usually require a 1:1 mapping (same input and output length). Seq2Seq handles N:M mapping.
- **Seq2Seq vs. Transformers:** Seq2Seq uses recurrence and a single context vector. Transformers use self-attention, no recurrence, and evaluate the entire sequence simultaneously, eliminating the bottleneck.

## Common Interview Traps
- **Trap:** Failing to mention the "Information Bottleneck" when asked why Attention was introduced. (Always link Seq2Seq's failure on long sequences to the birth of Attention).
- **Trap:** Thinking the context vector contains *all* hidden states. In basic Seq2Seq, it is strictly the *final* hidden state of the encoder.

## Python / SQL Syntax (if applicable)
```python
# High-level Keras Conceptual Setup (Basic Seq2Seq)
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

# ENCODER
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c] # <-- This is the Context Vector

# DECODER
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
# The context vector is passed as the initial state
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)
```

## Important Formula (if applicable)
- $h_t = f(W_{hx} x_t + W_{hh} h_{t-1} + b)$ (Encoder step)
- $C = h_T$ (The Context Vector $C$ is the hidden state at the final time step $T$)

## 45-Second Interview Answer
"A standard Seq2Seq model uses an Encoder-Decoder architecture to handle variable-length inputs and outputs. The Encoder reads the input sequence token by token and compresses it into a single, fixed-length context vector, which is typically its final hidden state. The Decoder then uses this vector to generate the output sequence. While revolutionary for tasks like translation, its major flaw is the 'information bottleneck'—forcing a long paragraph into one vector causes severe information loss. This specific limitation is exactly what prompted the invention of the Attention mechanism."

## Practice Questions:

### Q1:
**Objective:**
Build a minimal character-level Encoder-Decoder network to learn a simple sequence reversal or character translation task (e.g., mapping "hi" $\rightarrow$ "hello", "bye" $\rightarrow$ "ciao").

**Your Task:**
- Define an Encoder with an LSTM layer (set latent_dim = 64) that takes encoder_input_data and returns the final hidden and cell states (state_h, state_c).

- Define a Decoder with an LSTM layer and a Dense softmax layer that takes decoder_input_data, initialized with the encoder's states, to output character probabilities.

- Construct and compile the Model(inputs=[encoder_inputs, decoder_inputs], outputs=decoder_outputs) using 'rmsprop' optimizer and 'categorical_crossentropy' loss.

In [1]:
# Data:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

# Toy dataset: English to Spanish / simple mappings
input_texts = ['hi', 'bye', 'yes', 'no', 'thanks']
target_texts = ['\thola\n', '\tchau\n', '\tsi\n', '\tno\n', '\tgracias\n']

# Character sets
input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))
num_encoder_tokens = len(input_chars)
num_decoder_tokens = len(target_chars)

max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

input_token_index = {char: i for i, char in enumerate(input_chars)}
target_token_index = {char: i for i, char in enumerate(target_chars)}

# One-hot encoding
encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype='float32'
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype='float32'
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype='float32'
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

I0000 00:00:1789453356.879491   36495 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789453357.770283   36495 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789453359.434933   36495 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
from tensorflow.keras.layers import Dense, Input, LSTM
from tensorflow.keras.models import Model

latent_dim = 64

# 1. ENCODER
encoder_inputs = Input(shape=(None, num_encoder_tokens), name="encoder_inputs")
encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

# Context Vector: Final hidden and cell states
encoder_states = [state_h, state_c]

# 2. DECODER
decoder_inputs = Input(shape=(None, num_decoder_tokens), name="decoder_inputs")
decoder_lstm = LSTM(
    latent_dim, return_sequences=True, return_state=True, name="decoder_lstm"
)
# Pass context vector as the initial state
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

# 3. OUTPUT PROJECTION
decoder_dense = Dense(
    num_decoder_tokens, activation="softmax", name="output_dense"
)
decoder_outputs = decoder_dense(decoder_outputs)

# 4. MODEL COMPILATION
model = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs,
    name="seq2seq_model",
)
model.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

#### Key Interview Takeaways:
- **`return_state=True` on Encoder:** Extracts $[h_T, c_T]$, capturing the final sequence summary.
- **`initial_state=encoder_states` on Decoder:** Injects the context vector to seed generation.
- **`return_sequences=True` on Decoder:** Ensures a prediction is emitted at every output time step.
- **Limitation:** The single `encoder_states` list forms a fixed-capacity bottleneck for long sequences.